In [242]:
import pandas as pd

In [246]:
df_parties_perf = pd.read_excel(r"D:\Projects\Election_kerala\kerala-election-analysis\data\raw\2016\Performance of Poltical Parties.xlsx")
df_candidates = pd.read_excel(r"D:\Projects\Election_kerala\kerala-election-analysis\data\raw\2016\List of Successful Candidates.xlsx")
df_parties_list = pd.read_excel(r"D:\Projects\Election_kerala\kerala-election-analysis\data\raw\2016\List Of Political Parties Participated.xlsx")

In [251]:
print(df_parties_perf.head())
print(df_candidates.head())
print(df_parties_list.head())

  Performance Of Political Parties                           Unnamed: 1  \
0                       Party Type                           Party Name   
1                                N               Bharatiya Janata Party   
2                                N                  Bahujan Samaj Party   
3                                N             Communist Party of India   
4                                N  Communist Party of India  (Marxist)   

  Unnamed: 2 Unnamed: 3 Unnamed: 4 Unnamed: 5         Unnamed: 6  Unnamed: 7  \
0  Contested        Won  Forfitted      Votes  Total Valid Votes  Votes in %   
1         98          1         62    2129726           20232563   10.526229   
2         74          0         74      47638           20232563    0.235452   
3         25         19          0    1643878           20232563    8.124912   
4         84         58          0    5365472           20232563   26.518993   

                            Unnamed: 8                  Unnamed: 9  

In [257]:
print("Perfomance of political party: ",df_parties_perf.columns)
print("List of successfull candidates: ",df_candidates.columns)
print("List of Political parties participated: ",df_parties_list.columns)

Perfomance of political party:  Index(['Performance Of Political Parties', 'Unnamed: 1', 'Unnamed: 2',
       'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7',
       'Unnamed: 8', 'Unnamed: 9'],
      dtype='str')
List of successfull candidates:  Index(['List OF Successful Candidates', 'Unnamed: 1', 'Unnamed: 2',
       'Unnamed: 3', 'Unnamed: 4'],
      dtype='str')
List of Political parties participated:  Index(['List OF Participating Political Parties', 'Unnamed: 1', 'Unnamed: 2'], dtype='str')


## Clean

### Performance of Political Parties

In [261]:
df1 = df_parties_perf.copy()

df1.columns = df1.iloc[0]
df1 = df1[1:].reset_index(drop=True)

df1.columns = [
    "PARTY_TYPE",
    "PARTY_NAME",
    "CONTESTED",
    "WON",
    "FORFEITED",
    "VOTES",
    "TOTAL_VALID_VOTES",
    "VOTE_SHARE_PCT",
    "VALID_VOTES_IN_SEATS",
    "VOTE_PCT_IN_SEATS"
]

df1.to_csv(r"D:\Projects\Election_kerala\kerala-election-analysis\data\clean\2016\clean_parties_performance.csv", index=False)

In [263]:
df1.head()

,PARTY_TYPE,PARTY_NAME,CONTESTED,WON,FORFEITED,VOTES,TOTAL_VALID_VOTES,VOTE_SHARE_PCT,VALID_VOTES_IN_SEATS,VOTE_PCT_IN_SEATS
0,N,Bharatiya Janata Party,98,1,62,2129726,20232563,10.526229,14154420,15.046367
1,N,Bahujan Samaj Party,74,0,74,47638,20232563,0.235452,10671356,0.44641
2,N,Communist Party of India,25,19,0,1643878,20232563,8.124912,3668580,44.809654
3,N,Communist Party of India (Marxist),84,58,0,5365472,20232563,26.518993,12157773,44.13203
4,N,Indian National Congress,87,22,0,4794793,20232563,23.698396,12662156,37.867114


### Successful Candidates

In [262]:
df2 = df_candidates.copy()

df2.columns = df2.iloc[0]
df2 = df2[1:].reset_index(drop=True)


df2.columns = [
    "CONSTITUENCY",
    "AC_NO",
    "WINNER",
    "SEX",
    "PARTY"
]


df2["PARTY"] = df2["PARTY"].str.strip()


df2["AC_NO"] = pd.to_numeric(df2["AC_NO"], errors="coerce")


df2.to_csv(r"D:\Projects\Election_kerala\kerala-election-analysis\data\clean\2016\clean_successful_candidates.csv", index=False)

df2.head()

,CONSTITUENCY,AC_NO,WINNER,SEX,PARTY
0,Manjeshwar,1,P B Abdul Razak,M,IUML
1,Kasaragod,2,N A Nellikkunnu,M,IUML
2,Udma,3,K Kunhiraman,M,CPM
3,Kanhangad,4,E.Chandrashekaran,M,CPI
4,Trikaripur,5,M.Rajagopalan,M,CPM


### Political Parties Participated

In [265]:
df3 = df_parties_list.copy()

df3.columns = df3.iloc[0]
df3 = df3[1:].reset_index(drop=True)

df3.columns = [
    "PARTY_TYPE",
    "PARTY_NAME",
    "PARTY_ABBREVIATION"
]


df3["PARTY_NAME"] = df3["PARTY_NAME"].str.strip()
df3["PARTY_TYPE"] = df3["PARTY_TYPE"].str.strip()


df3.to_csv(r"D:\Projects\Election_kerala\kerala-election-analysis\data\clean\2016\clean_parties_list.csv", index=False)

df3.head()

,PARTY_TYPE,PARTY_NAME,PARTY_ABBREVIATION
0,N,Bharatiya Janata Party,BJP
1,N,Bahujan Samaj Party,BSP
2,N,Communist Party of India,CPI
3,N,Communist Party of India (Marxist),CPM
4,N,Indian National Congress,INC


## Preprocessing + Merging

In [267]:
TOTAL_CONSTITUENCIES = df2["CONSTITUENCY"].nunique()
TOTAL_CONSTITUENCIES

140

In [268]:
TOTAL_CONTESTANTS = df1["CONTESTED"].sum()

TOTAL_CONTESTANTS

1343

In [269]:
MIN_CONTESTANTS = df1["CONTESTED"].min()
MAX_CONTESTANTS = df1["CONTESTED"].max()

AVERAGE_CONTESTANTS_PER_CONSTITUENCY = TOTAL_CONTESTANTS / TOTAL_CONSTITUENCIES

MIN_CONTESTANTS, MAX_CONTESTANTS, AVERAGE_CONTESTANTS_PER_CONSTITUENCY

(1, 438, 9.592857142857143)

In [270]:
df_merged = df1.merge(
    df3[["PARTY_NAME", "PARTY_TYPE"]],
    on="PARTY_NAME",
    how="left",
    suffixes=("", "_FROM_LIST")
)

# PARTY_TYPE from df3
df_merged["PARTY_TYPE"] = df_merged["PARTY_TYPE_FROM_LIST"]

# Droping extra column
df_merged = df_merged.drop(columns=["PARTY_TYPE_FROM_LIST"])

df_merged.head()

,PARTY_TYPE,PARTY_NAME,CONTESTED,WON,FORFEITED,VOTES,TOTAL_VALID_VOTES,VOTE_SHARE_PCT,VALID_VOTES_IN_SEATS,VOTE_PCT_IN_SEATS
0,N,Bharatiya Janata Party,98,1,62,2129726,20232563,10.526229,14154420,15.046367
1,N,Bahujan Samaj Party,74,0,74,47638,20232563,0.235452,10671356,0.44641
2,N,Communist Party of India,25,19,0,1643878,20232563,8.124912,3668580,44.809654
3,N,Communist Party of India (Marxist),84,58,0,5365472,20232563,26.518993,12157773,44.13203
4,N,Indian National Congress,87,22,0,4794793,20232563,23.698396,12662156,37.867114


In [271]:
df_final = df_merged.copy()

df_final["TOTAL_CONSTITUENCIES"] = TOTAL_CONSTITUENCIES
df_final["TOTAL_CONTESTANTS"] = TOTAL_CONTESTANTS
df_final["AVERAGE_CONTESTANTS_PER_CONSTITUENCY"] = AVERAGE_CONTESTANTS_PER_CONSTITUENCY
df_final["MINIMUM_CONTESTANTS"] = MIN_CONTESTANTS
df_final["MAXIMUM_CONTESTANTS"] = MAX_CONTESTANTS

df_final.head()

,PARTY_TYPE,PARTY_NAME,CONTESTED,WON,FORFEITED,VOTES,TOTAL_VALID_VOTES,VOTE_SHARE_PCT,VALID_VOTES_IN_SEATS,VOTE_PCT_IN_SEATS,TOTAL_CONSTITUENCIES,TOTAL_CONTESTANTS,AVERAGE_CONTESTANTS_PER_CONSTITUENCY,MINIMUM_CONTESTANTS,MAXIMUM_CONTESTANTS
0,N,Bharatiya Janata Party,98,1,62,2129726,20232563,10.526229,14154420,15.046367,140,1343,9.592857,1,438
1,N,Bahujan Samaj Party,74,0,74,47638,20232563,0.235452,10671356,0.44641,140,1343,9.592857,1,438
2,N,Communist Party of India,25,19,0,1643878,20232563,8.124912,3668580,44.809654,140,1343,9.592857,1,438
3,N,Communist Party of India (Marxist),84,58,0,5365472,20232563,26.518993,12157773,44.13203,140,1343,9.592857,1,438
4,N,Indian National Congress,87,22,0,4794793,20232563,23.698396,12662156,37.867114,140,1343,9.592857,1,438


In [272]:
df_final = df_final[
    [
        "PARTY_TYPE",
        "PARTY_NAME",
        "CONTESTED",
        "WON",
        "FORFEITED",
        "VOTES",
        "VOTE_SHARE_PCT",
        "TOTAL_VALID_VOTES",
        "TOTAL_CONSTITUENCIES",
        "TOTAL_CONTESTANTS",
        "AVERAGE_CONTESTANTS_PER_CONSTITUENCY",
        "MINIMUM_CONTESTANTS",
        "MAXIMUM_CONTESTANTS"
    ]
]

df_final.head()

,PARTY_TYPE,PARTY_NAME,CONTESTED,WON,FORFEITED,VOTES,VOTE_SHARE_PCT,TOTAL_VALID_VOTES,TOTAL_CONSTITUENCIES,TOTAL_CONTESTANTS,AVERAGE_CONTESTANTS_PER_CONSTITUENCY,MINIMUM_CONTESTANTS,MAXIMUM_CONTESTANTS
0,N,Bharatiya Janata Party,98,1,62,2129726,10.526229,20232563,140,1343,9.592857,1,438
1,N,Bahujan Samaj Party,74,0,74,47638,0.235452,20232563,140,1343,9.592857,1,438
2,N,Communist Party of India,25,19,0,1643878,8.124912,20232563,140,1343,9.592857,1,438
3,N,Communist Party of India (Marxist),84,58,0,5365472,26.518993,20232563,140,1343,9.592857,1,438
4,N,Indian National Congress,87,22,0,4794793,23.698396,20232563,140,1343,9.592857,1,438


In [273]:
summary_rows = []

for ptype in df_final["PARTY_TYPE"].dropna().unique():
    temp = df_final[df_final["PARTY_TYPE"] == ptype]

    summary = {
        "PARTY_TYPE": ptype,
        "PARTY_NAME": f"{ptype}_TOTAL",
        "CONTESTED": temp["CONTESTED"].sum(),
        "WON": temp["WON"].sum(),
        "FORFEITED": temp["FORFEITED"].sum(),
        "VOTES": temp["VOTES"].sum(),
        "VOTE_SHARE_PCT": temp["VOTE_SHARE_PCT"].sum(),
        "TOTAL_VALID_VOTES": temp["TOTAL_VALID_VOTES"].max(),
        "TOTAL_CONSTITUENCIES": TOTAL_CONSTITUENCIES,
        "TOTAL_CONTESTANTS": TOTAL_CONTESTANTS,
        "AVERAGE_CONTESTANTS_PER_CONSTITUENCY": AVERAGE_CONTESTANTS_PER_CONSTITUENCY,
        "MINIMUM_CONTESTANTS": MIN_CONTESTANTS,
        "MAXIMUM_CONTESTANTS": MAX_CONTESTANTS
    }

    summary_rows.append(summary)

# TOTAL row
total_summary = {
    "PARTY_TYPE": "TOTAL",
    "PARTY_NAME": "ALL_PARTIES",
    "CONTESTED": df_final["CONTESTED"].sum(),
    "WON": df_final["WON"].sum(),
    "FORFEITED": df_final["FORFEITED"].sum(),
    "VOTES": df_final["VOTES"].sum(),
    "VOTE_SHARE_PCT": df_final["VOTE_SHARE_PCT"].sum(),
    "TOTAL_VALID_VOTES": df_final["TOTAL_VALID_VOTES"].max(),
    "TOTAL_CONSTITUENCIES": TOTAL_CONSTITUENCIES,
    "TOTAL_CONTESTANTS": TOTAL_CONTESTANTS,
    "AVERAGE_CONTESTANTS_PER_CONSTITUENCY": AVERAGE_CONTESTANTS_PER_CONSTITUENCY,
    "MINIMUM_CONTESTANTS": MIN_CONTESTANTS,
    "MAXIMUM_CONTESTANTS": MAX_CONTESTANTS
}

summary_rows.append(total_summary)

df_summary = pd.DataFrame(summary_rows)

In [274]:
df_summary

,PARTY_TYPE,PARTY_NAME,CONTESTED,WON,FORFEITED,VOTES,VOTE_SHARE_PCT,TOTAL_VALID_VOTES,TOTAL_CONSTITUENCIES,TOTAL_CONTESTANTS,AVERAGE_CONTESTANTS_PER_CONSTITUENCY,MINIMUM_CONTESTANTS,MAXIMUM_CONTESTANTS
0,N,N_TOTAL,372,102,136,14218915,70.277379,20232563,140,1343,9.592857,1,438
1,O,O_TOTAL,39,0,33,340234,1.681616,20232563,140,1343,9.592857,1,438
2,S,S_TOTAL,48,27,0,2813927,13.907912,20232563,140,1343,9.592857,1,438
3,U,U_TOTAL,306,5,280,1685039,8.328352,20232563,140,1343,9.592857,1,438
4,Z,Z_TOTAL,438,6,424,1067203,5.274680,20232563,140,1343,9.592857,1,438
5,Z1,Z1_TOTAL,140,0,140,107245,0.530061,20232563,140,1343,9.592857,1,438
6,TOTAL,ALL_PARTIES,1343,140,1013,20232563,100.000000,20232563,140,1343,9.592857,1,438


In [276]:
df_final_clean = df_merged.copy()

df_final_clean = df_final_clean[
    [
        "PARTY_TYPE",
        "PARTY_NAME",
        "CONTESTED",
        "WON",
        "FORFEITED",
        "VOTES",
        "VOTE_SHARE_PCT",
        "TOTAL_VALID_VOTES"
    ]
]

In [277]:
df_final_clean["PARTY_NAME"] = (
    df_final_clean["PARTY_NAME"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [278]:
summary_rows = []

summary_types = ["N", "S", "O", "U"]

for p in summary_types:
    temp = df_final_clean[df_final_clean["PARTY_TYPE"] == p]

    summary_rows.append({
        "PARTY_TYPE": p,
        "PARTY_NAME": f"{p}_TOTAL",
        "CONTESTED": temp["CONTESTED"].sum(),
        "WON": temp["WON"].sum(),
        "FORFEITED": temp["FORFEITED"].sum(),
        "VOTES": temp["VOTES"].sum(),
        "VOTE_SHARE_PCT": temp["VOTE_SHARE_PCT"].sum(),
        "TOTAL_VALID_VOTES": temp["TOTAL_VALID_VOTES"].max()
    })

# Z + Z1 combined
temp_z = df_final_clean[df_final_clean["PARTY_TYPE"].isin(["Z", "Z1"])]

summary_rows.append({
    "PARTY_TYPE": "Z",
    "PARTY_NAME": "Z_TOTAL",
    "CONTESTED": temp_z["CONTESTED"].sum(),
    "WON": temp_z["WON"].sum(),
    "FORFEITED": temp_z["FORFEITED"].sum(),
    "VOTES": temp_z["VOTES"].sum(),
    "VOTE_SHARE_PCT": temp_z["VOTE_SHARE_PCT"].sum(),
    "TOTAL_VALID_VOTES": temp_z["TOTAL_VALID_VOTES"].max()
})

df_summary = pd.DataFrame(summary_rows)

In [279]:
for col, val in {
    "TOTAL_CONSTITUENCIES": TOTAL_CONSTITUENCIES,
    "TOTAL_CONTESTANTS": TOTAL_CONTESTANTS,
    "AVERAGE_CONTESTANTS_PER_CONSTITUENCY": AVERAGE_CONTESTANTS_PER_CONSTITUENCY,
    "MINIMUM_CONTESTANTS": MIN_CONTESTANTS,
    "MAXIMUM_CONTESTANTS": MAX_CONTESTANTS
}.items():
    df_summary[col] = val

In [280]:
df_output = pd.concat([df_final, df_summary], ignore_index=True)

df_output.to_csv(r"D:\Projects\Election_kerala\kerala-election-analysis\data\preprocessed\2016\2016_election_summary.csv", index=False)

In [281]:
df_output

,PARTY_TYPE,PARTY_NAME,CONTESTED,WON,FORFEITED,VOTES,VOTE_SHARE_PCT,TOTAL_VALID_VOTES,TOTAL_CONSTITUENCIES,TOTAL_CONTESTANTS,AVERAGE_CONTESTANTS_PER_CONSTITUENCY,MINIMUM_CONTESTANTS,MAXIMUM_CONTESTANTS
0,N,Bharatiya Janata Party,98,1,62,2129726,10.526229,20232563,140,1343,9.592857,1,438
1,N,Bahujan Samaj Party,74,0,74,47638,0.235452,20232563,140,1343,9.592857,1,438
2,N,Communist Party of India,25,19,0,1643878,8.124912,20232563,140,1343,9.592857,1,438
3,N,Communist Party of India (Marxist),84,58,0,5365472,26.518993,20232563,140,1343,9.592857,1,438
4,N,Indian National Congress,87,22,0,4794793,23.698396,20232563,140,1343,9.592857,1,438
5,N,Nationalist Congress Party,4,2,0,237408,1.173396,20232563,140,1343,9.592857,1,438
6,O,All India Anna Dravida Munnetra Kazhagam,7,0,7,33440,0.165278,20232563,140,1343,9.592857,1,438
7,O,Janata Dal (United),7,0,1,296585,1.46588,20232563,140,1343,9.592857,1,438
8,O,Shivsena,16,0,16,4952,0.024475,20232563,140,1343,9.592857,1,438
9,O,Samajwadi Party,9,0,9,5257,0.025983,20232563,140,1343,9.592857,1,438


In [282]:
import numpy as np

cols_extra = [
    "TOTAL_CONSTITUENCIES",
    "TOTAL_CONTESTANTS",
    "AVERAGE_CONTESTANTS_PER_CONSTITUENCY",
    "MINIMUM_CONTESTANTS",
    "MAXIMUM_CONTESTANTS"
]

# create missing columns in party df if not exist
for col in cols_extra:
    if col not in df_final_clean.columns:
        df_final_clean[col] = np.nan

# ensure summary has same columns
for col in cols_extra:
    if col not in df_summary.columns:
        df_summary[col] = np.nan

# reorder columns (final schema)
final_cols = [
    "PARTY_TYPE", "PARTY_NAME",
    "CONTESTED", "WON", "FORFEITED",
    "VOTES", "VOTE_SHARE_PCT",
    "TOTAL_VALID_VOTES",
    "TOTAL_CONSTITUENCIES",
    "TOTAL_CONTESTANTS",
    "AVERAGE_CONTESTANTS_PER_CONSTITUENCY",
    "MINIMUM_CONTESTANTS",
    "MAXIMUM_CONTESTANTS"
]

df_final_clean = df_final_clean[final_cols]
df_summary = df_summary[final_cols]

In [283]:
df_summary

,PARTY_TYPE,PARTY_NAME,CONTESTED,WON,FORFEITED,VOTES,VOTE_SHARE_PCT,TOTAL_VALID_VOTES,TOTAL_CONSTITUENCIES,TOTAL_CONTESTANTS,AVERAGE_CONTESTANTS_PER_CONSTITUENCY,MINIMUM_CONTESTANTS,MAXIMUM_CONTESTANTS
0,N,N_TOTAL,372,102,136,14218915,70.277379,20232563,140,1343,9.592857,1,438
1,S,S_TOTAL,48,27,0,2813927,13.907912,20232563,140,1343,9.592857,1,438
2,O,O_TOTAL,39,0,33,340234,1.681616,20232563,140,1343,9.592857,1,438
3,U,U_TOTAL,306,5,280,1685039,8.328352,20232563,140,1343,9.592857,1,438
4,Z,Z_TOTAL,578,6,564,1174448,5.804742,20232563,140,1343,9.592857,1,438


In [284]:
df_output = pd.concat([df_final_clean, df_summary], ignore_index=True)

df_output.to_csv(r"D:\Projects\Election_kerala\kerala-election-analysis\data\preprocessed\2016\2016_Party_type_summary.csv", index=False)